# Notebook About IFrame/Keyframe and FFmpeg

ffprobe get keyframes only
- the use of csv is for easier human reading, you can use `json` too
```bash
ffprobe  -v error -skip_frame nokey -select_streams v:0 -show_frames -show_entries frame=pict_type,pts_time -of csv ./tracks/singking.webm
```
- the command will only find video keyframes, for audio, more handling is needed, likely no keyframe and we can cut anywhere

Example output format
```
frame,5.120000,I
frame,10.240000,I
```

In [1]:
import subprocess
ffmpeg_path = "C:\\Partitions\\G\\youtube_dlp\\ffmpeg.exe"
ffprobe_path = "C:\\Partitions\\G\\youtube_dlp\\ffprobe.exe"

In [11]:
result = subprocess.run([ffprobe_path, '-v', 'error', '-skip_frame', 'nokey', '-select_streams', 'v:0', '-show_frames', '-show_entries', 'frame=pict_type,pts_time', '-of', 'json', './tracks/singking.webm'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

In [16]:
import json

json.loads(result.stdout)['frames'][10:20]

[{'pts_time': '46.080000', 'pict_type': 'I'},
 {'pts_time': '51.200000', 'pict_type': 'I'},
 {'pts_time': '52.240000', 'pict_type': 'I'},
 {'pts_time': '56.320000', 'pict_type': 'I'},
 {'pts_time': '59.920000', 'pict_type': 'I'},
 {'pts_time': '61.440000', 'pict_type': 'I'},
 {'pts_time': '66.560000', 'pict_type': 'I'},
 {'pts_time': '67.880000', 'pict_type': 'I'},
 {'pts_time': '71.680000', 'pict_type': 'I'},
 {'pts_time': '74.360000', 'pict_type': 'I'}]

### ffmpeg behavior with keyframes
NOTE: the testing is done on ffmpeg Windows and VLC media player, which may be different from Linux ffmpeg and web browser used in the web application.

Assuming 0.00 and 10.24 are frames to be cut, with a keyframe at 5.12
The `-ss` is set after `-i`, it decodes the input file until the cut point, output seeking
- setting `-ss` to 0.00 will cut at the first keyframe (0.00)
- setting `-ss` to 0.01 will cause black frame/freeze until the next keyframe (5.12)
- setting `-to` to 10.25 (10.23) likely won't cause significant issues

The `-ss` is set before `-i`, it seeks to the cut point before decoding, input seeking
- setting `-ss` to 0.01 creates similar issue as output seeking, but it's because the output timestamp starts at negative value, not because it starts at a non-keyframe
- using the option `avoid_negative_ts make_zero` can fix the issue by making the output timestamp starts at zero
- this makes ffmpeg behave like it can cut at non-keyframe, but it's actually cutting at the nearest keyframe before the cut point

Example command from losslesscut
```
ffmpeg -hide_banner -ss 5.74598 -i C:\Users\hubcc\Documents\Projects\karaoke\tracks\singking.mp4 -t 2.99790 -avoid_negative_ts make_zero -map 0:0 -c:0 copy -map 0:1 -c:1 copy -map_metadata 0 -movflags +faststart -default_mode infer_no_subs -ignore_unknown -f mp4 -y C:\Users\hubcc\Documents\Projects\karaoke\tracks\singking-00.00.05.746-00.00.08.744.mp4
```

- `-movflags +faststart` is used to move the moov atom to the beginning of the file, which allows for faster playback start
- `avoid_negative_ts make_zero` is used to avoid negative timestamps in the output file, which can cause issues with some players
    - suppose we cut -ss at 0.01, without `avoid_negative_ts make_zero`, the output file will have negative timestamps and starts at `-0.01`, depending on the length where the timestamp becomes zero, it may cause issues/black frame

Production ready video cutting command.
- first use the previous command to get the keyframes and their timestamps
```bash
ffmpeg -hide_banner -ss $start_time -i input.mp4 -t $end_time-$start_time -avoid_negative_ts make_zero -movflags +faststart -c copy output.mp4
```
- this should produce a mp4 file that has both the audio and video tracks

DO NOT use anything else other than `-c copy` especially `-c:v copy`, `-c:a some_codec` is acceptable, for example `-c:v libx264|libx265|vp9` etc is **strictly forbidden** when implementing this

The similar production command works well with audio formats too, even though there's no keyframe, it can still cut at these points without issues, producing a synchronized vocal sidecar. Just replace the input and output file extensions to the desired audio format.

### Subtitle/Lyrics/JSON handling

In [3]:
import srt
import pylrc

In [119]:
with open('./tracks/facelikeyours.lrc', 'r', encoding='utf-8') as f:
    lrc_content = f.read()

lrc = pylrc.parse(lrc_content)

In [6]:
dir(lrc[0])

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_check',
 'addHours',
 'addMillis',
 'addMinutes',
 'addSeconds',
 'hours',
 'milliseconds',
 'minutes',
 'seconds',
 'shift',
 'text',
 'time']

In [8]:
import inspect

lrc[0].shift.__code__.co_varnames

('self', 'minutes', 'seconds', 'milliseconds')

In [106]:
import math
from collections.abc import Iterable

def shift_lrc(lrc: Iterable, start: float, end: float):
    """Shift the timing of LRC lines within a specified time range."""
    fractional, second = math.modf(start)
    milliseconds = round(fractional * 1000)
    for line in lrc:
        if line.time - start < 0 or line.time - end > 0:
            line.time = 0 # set unqualified time to 0 for easier filtering later
        else:
            line.shift(seconds=-second, milliseconds=-milliseconds)
            print(f"Shifted line: {line.text}, new time: {line.time}")
    return pylrc.classes.Lyrics([line for line in lrc if line.time > 0]) # type: ignore

In [ ]:
shifted_lrc = shift_lrc(lrc, 10.23, 140.183) # this function WILL mutate the original lrc object, only run it once or add functions to make a copy first
shifted_lrc.toLRC()

In [122]:
shifted_lrc.toLRC()

"[00:00.06]Know you peaked in high school but it still gets to your head/你的姿色迷人，灵魂却失色，昔日校园荣光仍在梦中闪烁，\n[00:03.48]Hot can only take you as far as my bed/但那种吸引力仅存于刹那间的亲近之间。\n[00:06.23]Well, at least we got that/至少，在那个瞬间，我们共享了彼此的世界。\n[00:10.10]It's not your fault/这不是你的过错，\n[00:11.67]Your looks are ridiculous/你亮丽的外表令人无法忽视，\n[00:13.84]Just your fault/却也成为沉重的枷锁。\n[00:15.24]You're not living up to it/你未曾照亮期望中的璀璨星辰。\n[00:17.25]Oh my god/哦，我的天，\n[00:18.56]It's right at your fingertips/一切皆在指间流转\n[00:20.71]Oh oh oh/却终成一场空，\n[00:23.40]What a waste of a face like yours/哎，多么悲惋，\n[00:26.91]What a shame you were nothing more/绝代风华竟付流水，\n[00:30.22]Than a night to remember a hand-me-down sweater/可惜，不过是回忆中一件传承的毛衣\n[00:34.33]A future as dry as the desert/遥望的未来荒芜如烈日下的沙丘。\n[00:37.10]With your hands on my body I'm swearing you're my type/纵使肌肤相亲，我仍坚信你符合我的理想，\n[00:40.57]When you open up your mouth, oh god, I wanna die/但你启唇的刹那，哦，我心中祈求遁入静寂。\n[00:44.27]What a waste of a face like yours/何其痛惜，如斯容颜的枉费，\n[00:50.65]You'r

In [123]:
with open('./tracks/facelikeyours_shifted.lrc', 'w', encoding='utf-8') as f:
    f.write(shifted_lrc.toLRC())

In [141]:
with open('./tracks/facelikeyours.srt', 'r', encoding='utf-8') as f:
    subs = list(srt.parse(f.read()))

In [137]:
import datetime
def shift_srt(srt: list, start: float, end: float):
    """Shift the timing of SRT subtitles within a specified time range."""
    fractional, second = math.modf(start)
    milliseconds = round(fractional * 1000)
    for sub in srt:
        if sub.start.total_seconds() - start < 0 or sub.start.total_seconds() - end > 0:
            sub.start = sub.end = datetime.timedelta(0) # set unqualified time to 0 for easier filtering later
        else:
            sub.start -= datetime.timedelta(seconds=second, milliseconds=milliseconds)
            sub.end -= datetime.timedelta(seconds=second, milliseconds=milliseconds)
            print(f"Shifted subtitle: {sub.content}, new start time: {sub.start}, new end time: {sub.end}")
    return [sub for sub in srt if sub.start.total_seconds() > 0] # type: ignore

In [ ]:
shifted_srt = shift_srt(subs, 10.23, 140.183) # similar to shift_lrc, this function will mutate the original srt list, only run it once or add functions to make a copy first
print(srt.compose(shifted_srt))

In [167]:
# Whisperx JSON file
import json
with open('./tracks/facelikeyours.json', 'r', encoding='utf-8') as f:
    aligned = json.load(f)

In [158]:
aligned

[{'start': 0, 'end': 0, 'text': "You're so unoriginal", 'words': []},
 {'start': 0, 'end': 0, 'text': 'For leaving me on read', 'words': []},
 {'start': 0,
  'end': 0,
  'text': 'Know you peaked in high school but it still gets to your head',
  'words': [{'word': 'Hot', 'start': 0, 'end': 0, 'score': 0.492, 'words': []},
   {'word': 'Hot', 'start': 0, 'end': 0, 'score': 0.492, 'words': []},
   {'word': 'can',
    'start': 3.8520000000000003,
    'end': 3.991999999999999,
    'score': 0.864}]},
 {'start': 3.6319999999999997,
  'end': 6.352,
  'text': 'Hot can only take you as far as my bed',
  'words': [{'word': 'Hot', 'start': 0, 'end': 0, 'score': 0.492, 'words': []},
   {'word': 'can',
    'start': 3.8520000000000003,
    'end': 3.991999999999999,
    'score': 0.864},
   {'word': 'only', 'start': 14.262, 'end': 14.462, 'score': 0.963},
   {'word': 'take', 'start': 14.522, 'end': 14.922, 'score': 0.848},
   {'word': 'you', 'start': 14.962, 'end': 15.322, 'score': 0.776},
   {'word': '

In [176]:
import functools
def round_output(value, decimals=3):
    @functools.wraps(value)
    def wrapper(*args, **kwargs):
        r1, r2 = value(*args, **kwargs)
        if isinstance(r1, (int, float)):
            r1 = round(r1, decimals)
        if isinstance(r2, (int, float)):
            r2 = round(r2, decimals)
        return r1, r2
    return wrapper
    

In [195]:
@round_output
def modify_start_end(start,end, shift_start, shift_end) -> tuple[float | None, float | None]:
    if end < shift_start or start > shift_end:
        # should be filtered
        return None, None
    # compute clipped/shifted start and end within the [shift_start, shift_end] window
    new_start = max(start - shift_start, 0)
    new_end = max(min(end, shift_end) - shift_start, 0)
    return new_start, new_end

In [196]:
modify_start_end(15, 25, 8, 140.183)

(7, 17)

In [185]:
def shift_whisperx(aligned: dict, start: float, end: float):
    aligned_new = []
    for idx, segment in enumerate(aligned):
        new_start, new_end = modify_start_end(segment['start'], segment['end'], start, end)
        if not new_start and not new_end:
            continue
        aligned_new.append({'start': new_start, 'end': new_end, 'text': segment['text'], 'words': []})
        for word in segment['words']:
            word_start, word_end = modify_start_end(word['start'], word['end'], start, end)
            if not word_start and not word_end:
                continue
            aligned_new[-1]['words'].append({'start': word_start, 'end': word_end, 'text': word['word']})
    return aligned_new

In [ ]:
whisperx_shifted = shift_whisperx(aligned, 10.23, 140.183)
whisperx_shifted

### Tasks for I-Frame video editing
Backend send a list of suitable keyframes to the frontend
User in frontend will select the start and end time for cutting, it can be anything, the backend will validate it
- start time: shifted to the nearest keyframe before it
- end time: shifted to the nearest keyframe after it

Perform lossless cut with ffmpeg on the original video file
If there are sidecar vocal files: perform lossless cut on the vocal files with the same start and end time, this should produce perfectly synchronized vocal sidecar
Run shift_lrc, shift_srt, shift_whisperx functions to shift the timestamps of the sidecar files.

Temporary files for all of these can be produced, but in the end, the editing should feel "in-place", so editing the original files destructively and replace with the new files. There should be a prompt in browser before user submit the edit request.
